# Stage 8 Explainer Notebook

This notebook mirrors Stage 8 scripts step-by-step:
- preflight checks
- topic runs
- lab runs
- optional Qdrant path
- deliverable verification


## 1) Environment and Helpers
Run this notebook from `red-book/src/stage-8`.

In [ ]:
from pathlib import Path
import subprocess
import sys
import os

ROOT = Path.cwd()
print('Working dir:', ROOT)

def run_cmd(cmd):
    print(f'\n>>> {cmd}')
    p = subprocess.run(cmd, shell=True, cwd=ROOT, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if p.returncode != 0:
        raise RuntimeError(f'Command failed ({p.returncode}): {cmd}')

def run_py(script):
    run_cmd(f'{sys.executable} {script}')


## 2) Preflight Checks

In [ ]:
run_cmd(f'{sys.executable} --version')
run_cmd(f'{sys.executable} -c "import numpy, sklearn; print(\"numpy/sklearn ready\")"')
run_cmd(f'{sys.executable} -c "import torch; print(\"torch\", torch.__version__, \"cuda_available\", torch.cuda.is_available())"')

## 3) Run Key Topic Scripts (Intermediate Path)

In [ ]:
topic_scripts = [
    'topic00_pytorch_cuda_tuning_intermediate.py',
    'topic01_dataset_quality_intermediate.py',
    'topic02_sft_intermediate.py',
    'topic03_lora_intermediate.py',
    'topic04_qlora_intermediate.py',
    'topic05_distill_intermediate.py',
    'topic06_tune_vs_rag_intermediate.py',
    'topic07_eval_regression_intermediate.py',
    'topic08_ops_observability_intermediate.py',
]

for s in topic_scripts:
    run_py(s)

## 4) Run Lab Set (Core)

In [ ]:
lab_scripts = [
    'lab01_instruction_tuning_baseline.py',
    'lab02_lora_qlora_comparison.py',
    'lab03_distillation_tradeoff_lab.py',
    'lab04_finetune_project_baseline_to_production.py',
]

for s in lab_scripts:
    run_py(s)

## 5) Optional: Run Qdrant Comparison Lab

In [ ]:
qdrant_url = 'http://localhost:6333/collections'

try:
    import urllib.request
    with urllib.request.urlopen(qdrant_url, timeout=1.0) as resp:
        ok = getattr(resp, 'status', 200) == 200
except Exception:
    ok = False

print('Qdrant available:', ok)
if ok:
    run_py('lab05_finetune_vs_rag_vs_hybrid_qdrant.py')
else:
    print('Skipping lab05 (Qdrant not reachable).')

## 6) Verify Required Deliverables

In [ ]:
# Verify core outputs (labs 1-4)
run_cmd('powershell -ExecutionPolicy Bypass -File .\verify_stage8_outputs.ps1')

In [ ]:
# Verify with optional Qdrant outputs
if ok:
    run_cmd('powershell -ExecutionPolicy Bypass -File .\verify_stage8_outputs.ps1 -IncludeQdrant')
else:
    print('Qdrant verification skipped because local server is unavailable.')

## 7) Inspect Output Files

In [ ]:
results = ROOT / 'results'
for p in sorted(results.glob('*')):
    size = p.stat().st_size
    print(f'{p.name:55} {size:>8} bytes')

## 8) Next-Step Reflection
Use generated reports to answer:
1. Which method won on quality?
2. Which method won on memory/latency?
3. What is your final promote/hold/rollback decision and why?